In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "enviroment_bj").exists():
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("project_root:", ROOT)

In [2]:

from pprint import pprint

import torch

from enviroment_bj import BlackjackConfig, BlackjackEnvironment, ObservationConfig, ACTION_ORDER
from model.encoder import BlackjackObservationEncoder


In [ ]:
# helper para crear env + helper de inspección

def make_env(profile, shoe, seed=11, **config_overrides):
    obs = ObservationConfig.for_profile(profile)
    config = BlackjackConfig(
        n_decks=1,
        shoe_penetration=1.0,
        observation=obs,
        **config_overrides,
    )
    env = BlackjackEnvironment(config=config, seed=seed)
    env.load_shoe(shoe, total_cards=len(shoe))
    return env

def inspect_encoded_response(profile, response, encoder):
    encoded = encoder(response)

    print("=" * 100)
    print("PROFILE:", profile)
    print("ACTION_ORDER:", ACTION_ORDER)
    print("observation keys:", sorted(response["observation"].keys()))
    print("table_rules keys:", sorted(response["table_rules"].keys()))
    print("action_mask:", response["action_mask"])
    print("action_mask tensor shape:", tuple(encoded["action_mask"].shape))
    print("state_vector shape:", tuple(encoded["state_vector"].shape))
    print("state_dim from encoder:", encoder.state_dim)
    print()

    print("module_dims:")
    pprint(dict(encoder.module_dims))
    print()

    print("module_slices:")
    pprint(dict(encoded["metadata"]["module_slices"]))
    print()

    print("module tensor shapes:")
    pprint({k: tuple(v.shape) for k, v in encoded["module_tensors"].items()})
    print()

    print("nonzero por modulo:")
    pprint({k: int(torch.count_nonzero(v)) for k, v in encoded["module_tensors"].items()})

    return encoded


In [4]:
#  perfil minimal_basic_strategy

env = make_env("minimal_basic_strategy", ["10", "6", "7", "10"])
response = env.reset()
encoder = BlackjackObservationEncoder.from_profile("minimal_basic_strategy")

encoded_min = inspect_encoded_response("minimal_basic_strategy", response, encoder)


PROFILE: minimal_basic_strategy
ACTION_ORDER: ('stand', 'hit', 'double', 'split', 'surrender', 'insurance')
observation keys: ['current_hand_is_soft', 'current_hand_total', 'dealer_upcard', 'dealer_upcard_value', 'hand_context', 'insurance_context', 'mode', 'profile']
table_rules keys: ['base_bet', 'blackjack_payout', 'dealer_hits_soft_17', 'double_after_split_allowed', 'double_allowed_on', 'hit_split_aces_allowed', 'insurance_allowed', 'max_hands_after_split', 'resplit_aces_allowed', 'split_rule', 'surrender_allowed']
action_mask: [1, 1, 1, 0, 1, 0]
action_mask tensor shape: (6,)
state_vector shape: (40,)
state_dim from encoder: 40

module_dims:
{'hand': 16, 'hand_context': 5, 'insurance': 2, 'rules': 17}

module_slices:
{'hand': (0, 16),
 'hand_context': (16, 21),
 'insurance': (21, 23),
 'rules': (23, 40)}

module tensor shapes:
{'hand': (16,), 'hand_context': (5,), 'insurance': (2,), 'rules': (17,)}

nonzero por modulo:
{'hand': 3, 'hand_context': 2, 'insurance': 0, 'rules': 9}


In [5]:


for name, tensor in encoded_min["module_tensors"].items():
    print(f"\n{name} | shape={tuple(tensor.shape)}")
    print(tensor)



hand | shape=(16,)
tensor([0.8095, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 1.0000, 0.0000,
        0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.5455])

hand_context | shape=(5,)
tensor([0.0000, 0.2000, 0.0000, 0.0000, 1.0000])

insurance | shape=(2,)
tensor([0., 0.])

rules | shape=(17,)
tensor([0.0000, 0.0000, 1.0000, 1.0000, 0.0000, 1.0000, 1.0000, 1.0000, 0.0000,
        0.0000, 0.0000, 1.0000, 0.7500, 0.1000, 0.5000, 0.0000, 0.0000])


In [6]:
# perfil table_realistic_default

env = make_env("table_realistic_default", ["8", "6", "8", "10", "3", "K", "2", "10"])
response = env.reset()
encoder = BlackjackObservationEncoder.from_profile("table_realistic_default")

encoded_default = inspect_encoded_response("table_realistic_default", response, encoder)


PROFILE: table_realistic_default
ACTION_ORDER: ('stand', 'hit', 'double', 'split', 'surrender', 'insurance')
observation keys: ['current_bet', 'current_hand_cards', 'dealer_upcard', 'dealer_upcard_value', 'discard_summary', 'hand_context', 'insurance_context', 'mode', 'observed_cards_history', 'other_player_hands_visible', 'profile', 'temporal_context']
table_rules keys: ['base_bet', 'blackjack_payout', 'dealer_hits_soft_17', 'double_after_split_allowed', 'double_allowed_on', 'hit_split_aces_allowed', 'insurance_allowed', 'max_hands_after_split', 'resplit_aces_allowed', 'split_rule', 'surrender_allowed']
action_mask: [1, 1, 1, 1, 1, 0]
action_mask tensor shape: (6,)
state_vector shape: (1089,)
state_dim from encoder: 1089

module_dims:
{'bet': 1,
 'discard_summary': 144,
 'hand': 185,
 'hand_context': 5,
 'insurance': 2,
 'observed_history': 14,
 'other_hands': 700,
 'rules': 17,
 'temporal': 21}

module_slices:
{'bet': (892, 893),
 'discard_summary': (924, 1068),
 'hand': (0, 185),
 '

In [7]:
# diseccionar el vector final por slices
# Esto sirve para verificar que el concatenado coincide con module_slices.

state = encoded_default["state_vector"]

for name, (start, end) in encoder.module_slices.items():
    chunk = state[start:end]
    module_tensor = encoded_default["module_tensors"].get(name)
    print(f"{name:20s} slice=({start:4d}, {end:4d}) shape={tuple(chunk.shape)}")
    if module_tensor is not None:
        print("  coincide con modulo:", torch.allclose(chunk, module_tensor))


hand                 slice=(   0,  185) shape=(185,)
  coincide con modulo: True
other_hands          slice=( 185,  885) shape=(700,)
  coincide con modulo: True
hand_context         slice=( 885,  890) shape=(5,)
  coincide con modulo: True
insurance            slice=( 890,  892) shape=(2,)
  coincide con modulo: True
bet                  slice=( 892,  893) shape=(1,)
  coincide con modulo: True
rules                slice=( 893,  910) shape=(17,)
  coincide con modulo: True
observed_history     slice=( 910,  924) shape=(14,)
  coincide con modulo: True
discard_summary      slice=( 924, 1068) shape=(144,)
  coincide con modulo: True
temporal             slice=(1068, 1089) shape=(21,)
  coincide con modulo: True


In [8]:
# interpretar dimensiones esperadas del perfil default

cfg = encoder.config

hand_dim = cfg.max_current_hand_cards * 13 + cfg.max_current_hand_cards + 3 + 13 + 1
other_hand_per_hand_dim = cfg.max_cards_per_hand * 13 + cfg.max_cards_per_hand + 7
other_hands_dim = cfg.max_other_hands * other_hand_per_hand_dim
observed_history_dim = 13 + 1
discard_summary_dim = 1 + 3 + cfg.max_recent_discard_cards * 13 + cfg.max_recent_discard_cards
temporal_dim = 21

expected = {
    "hand": hand_dim,
    "other_hands": other_hands_dim,
    "hand_context": 5,
    "insurance": 2,
    "bet": 1,
    "rules": 17,
    "observed_history": observed_history_dim,
    "discard_summary": discard_summary_dim,
    "temporal": temporal_dim,
}

print("expected dims:")
pprint(expected)

print("\nactual dims:")
pprint(dict(encoder.module_dims))

print("\nstate_dim esperado:", sum(expected.values()))
print("state_dim real:", encoder.state_dim)


expected dims:
{'bet': 1,
 'discard_summary': 144,
 'hand': 185,
 'hand_context': 5,
 'insurance': 2,
 'observed_history': 14,
 'other_hands': 700,
 'rules': 17,
 'temporal': 21}

actual dims:
{'bet': 1,
 'discard_summary': 144,
 'hand': 185,
 'hand_context': 5,
 'insurance': 2,
 'observed_history': 14,
 'other_hands': 700,
 'rules': 17,
 'temporal': 21}

state_dim esperado: 1089
state_dim real: 1089


In [9]:
# mirar la observacion cruda para entender de dónde sale cada tensor

pprint(response["observation"])


{'current_bet': 1.0,
 'current_hand_cards': ['8', '8'],
 'dealer_upcard': '6',
 'dealer_upcard_value': 6,
 'discard_summary': {'by_group': {'high': 0, 'low': 1, 'neutral': 2},
                     'observed_cards_count': 3,
                     'recent_cards': ['8', '6', '8']},
 'hand_context': {'current_hand_index': 0,
                  'first_decision_on_hand': True,
                  'from_split': False,
                  'n_player_hands': 1,
                  'split_aces': False},
 'insurance_context': {'insurance_bet': 0.0, 'insurance_offer_active': False},
 'mode': 'table_raw',
 'observed_cards_history': {'10': 0,
                            '2': 0,
                            '3': 0,
                            '4': 0,
                            '5': 0,
                            '6': 1,
                            '7': 0,
                            '8': 2,
                            '9': 0,
                            'A': 0,
                            'J': 0,
            

In [10]:
# escenario con split para que other_hands deje de estar casi vacio

env = make_env("table_realistic_default", ["8", "6", "8", "10", "3", "K", "2", "10"])
encoder = BlackjackObservationEncoder.from_profile("table_realistic_default")

start = env.reset()
after_split = env.step("split")

print("action_mask start:", start["action_mask"])
print("action_mask after_split:", after_split["action_mask"])

encoded_split = inspect_encoded_response("table_realistic_default / after split", after_split, encoder)


action_mask start: [1, 1, 1, 1, 1, 0]
action_mask after_split: [1, 1, 1, 0, 0, 0]
PROFILE: table_realistic_default / after split
ACTION_ORDER: ('stand', 'hit', 'double', 'split', 'surrender', 'insurance')
observation keys: ['current_bet', 'current_hand_cards', 'dealer_upcard', 'dealer_upcard_value', 'discard_summary', 'hand_context', 'insurance_context', 'mode', 'observed_cards_history', 'other_player_hands_visible', 'profile', 'temporal_context']
table_rules keys: ['base_bet', 'blackjack_payout', 'dealer_hits_soft_17', 'double_after_split_allowed', 'double_allowed_on', 'hit_split_aces_allowed', 'insurance_allowed', 'max_hands_after_split', 'resplit_aces_allowed', 'split_rule', 'surrender_allowed']
action_mask: [1, 1, 1, 0, 0, 0]
action_mask tensor shape: (6,)
state_vector shape: (1089,)
state_dim from encoder: 1089

module_dims:
{'bet': 1,
 'discard_summary': 144,
 'hand': 185,
 'hand_context': 5,
 'insurance': 2,
 'observed_history': 14,
 'other_hands': 700,
 'rules': 17,
 'temporal'

In [11]:
# inspección específica de other_hands y hand_context tras split

print("other_hands shape:", tuple(encoded_split["module_tensors"]["other_hands"].shape))
print(encoded_split["module_tensors"]["other_hands"])

print("\nhand_context:")
print(encoded_split["module_tensors"]["hand_context"])

print("\nobservacion hand_context cruda:")
pprint(after_split["observation"]["hand_context"])

print("\nobservacion other_player_hands_visible cruda:")
pprint(after_split["observation"]["other_player_hands_visible"])


other_hands shape: (700,)
tensor([0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 1.0000, 0.0000,
        0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 1.0000, 0.0000,
        0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000

In [12]:
#  fully_observable_sim

env = make_env("fully_observable_sim", ["10", "6", "7", "10", "5", "2"])
response = env.reset()
encoder = BlackjackObservationEncoder.from_profile("fully_observable_sim")

encoded_full = inspect_encoded_response("fully_observable_sim", response, encoder)


PROFILE: fully_observable_sim
ACTION_ORDER: ('stand', 'hit', 'double', 'split', 'surrender', 'insurance')
observation keys: ['current_bet', 'current_hand_cards', 'dealer_upcard', 'dealer_upcard_value', 'discard_summary', 'exact_shoe_composition', 'hand_context', 'insurance_context', 'mode', 'observed_cards_history', 'other_player_hands_visible', 'profile', 'temporal_context']
table_rules keys: ['base_bet', 'blackjack_payout', 'dealer_hits_soft_17', 'dealer_peeks_for_blackjack', 'double_after_split_allowed', 'double_allowed_on', 'hit_split_aces_allowed', 'insurance_allowed', 'max_hands_after_split', 'n_decks', 'resplit_aces_allowed', 'reward_mode', 'shoe_penetration', 'split_rule', 'surrender_allowed']
action_mask: [1, 1, 1, 0, 1, 0]
action_mask tensor shape: (6,)
state_vector shape: (1262,)
state_dim from encoder: 1262

module_dims:
{'bet': 1,
 'discard_summary': 144,
 'exact_shoe': 13,
 'hand': 185,
 'hand_context': 5,
 'insurance': 2,
 'observed_history': 14,
 'other_hands': 700,
 'r

In [13]:
# inspección del exact_shoe y temporal grande en fully_observable_sim

print("exact_shoe:")
print(encoded_full["module_tensors"]["exact_shoe"])
print("sum exact_shoe:", encoded_full["module_tensors"]["exact_shoe"].sum().item())

print("\ntemporal shape:", tuple(encoded_full["module_tensors"]["temporal"].shape))
print("temporal nonzero:", int(torch.count_nonzero(encoded_full["module_tensors"]["temporal"])))


exact_shoe:
tensor([0.0000, 0.5000, 0.0000, 0.0000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0000, 0.0000])
sum exact_shoe: 1.0

temporal shape: (181,)
temporal nonzero: 11


In [14]:
#  activar recent_actions en el encoder y en la observacion
# Aquí temporal pasa de 21 a 149 cuando max_recent_actions=8:
# 21 base + 8*15 + 8 = 149

obs = ObservationConfig.for_profile("table_realistic_default")
obs.obs_include_recent_actions = True
obs.obs_recent_actions_window = 8
obs.__post_init__()

env = BlackjackEnvironment(
    config=BlackjackConfig(n_decks=1, shoe_penetration=1.0, observation=obs),
    seed=11,
)
env.load_shoe(["10", "6", "7", "10", "10", "9", "5", "2", "10", "K", "8"], total_cards=11)

encoder = BlackjackObservationEncoder.from_profile(
    "table_realistic_default",
    encode_recent_actions=True,
    max_recent_actions=8,
)

start = env.reset()
end = env.step("stand")
next_round = env.reset()

for label, response in [("start", start), ("end", end), ("next_round", next_round)]:
    encoded = encoder(response)
    print("\n" + "=" * 80)
    print(label)
    print("temporal_context:")
    pprint(response["observation"]["temporal_context"])
    print("temporal shape:", tuple(encoded["module_tensors"]["temporal"].shape))
    print("state_vector shape:", tuple(encoded["state_vector"].shape))



start
temporal_context:
{'dealer_hands_seen_since_shuffle': 1,
 'dealer_hands_seen_total': 1,
 'estimated_shoe_progress': {'bucket': 'mid',
                             'fraction_used': 0.36363636363636365},
 'player_hands_seen_since_shuffle': 1,
 'player_hands_seen_total': 1,
 'recent_actions': [{'action': 'deal_round',
                     'actor': 'table',
                     'dealer_upcard': '6',
                     'player_hands': 1,
                     'round_index': 1,
                     'token': 'table:deal_round'}],
 'rounds_played_total': 1,
 'rounds_since_shuffle': 1,
 'shuffle_count': 2}
temporal shape: (149,)
state_vector shape: (1217,)

end
temporal_context:
{'dealer_hands_seen_since_shuffle': 1,
 'dealer_hands_seen_total': 1,
 'estimated_shoe_progress': {'bucket': 'mid',
                             'fraction_used': 0.45454545454545453},
 'player_hands_seen_since_shuffle': 1,
 'player_hands_seen_total': 1,
 'recent_actions': [{'action': 'deal_round',
              

In [15]:
# batch encoding -> [B, D]

env = make_env("minimal_basic_strategy", ["10", "6", "7", "10", "9", "5", "2", "10"])
first = env.reset()
second = env.step("stand")

encoder = BlackjackObservationEncoder.from_profile("minimal_basic_strategy")
batch = encoder.encode_batch([first, second])

print("state_vector batch shape:", tuple(batch["state_vector"].shape))
print("action_mask batch shape:", tuple(batch["action_mask"].shape))
print("hand batch shape:", tuple(batch["module_tensors"]["hand"].shape))
print("metadata:")
pprint(batch["metadata"])


state_vector batch shape: (2, 40)
action_mask batch shape: (2, 6)
hand batch shape: (2, 16)
metadata:
{'batch_size': 2,
 'items': [{'module_dims': {'hand': 16,
                            'hand_context': 5,
                            'insurance': 2,
                            'rules': 17},
            'module_slices': {'hand': (0, 16),
                              'hand_context': (16, 21),
                              'insurance': (21, 23),
                              'rules': (23, 40)},
            'observation_mode': 'basic_strategy',
            'observation_profile': 'minimal_basic_strategy',
            'profile': 'minimal_basic_strategy',
            'state_dim': 40},
           {'module_dims': {'hand': 16,
                            'hand_context': 5,
                            'insurance': 2,
                            'rules': 17},
            'module_slices': {'hand': (0, 16),
                              'hand_context': (16, 21),
                              'insu

In [ ]:
# sequence batch -> [B, T, D] con padding

encoder = BlackjackObservationEncoder.from_profile("minimal_basic_strategy")

env_a = make_env("minimal_basic_strategy", ["10", "6", "7", "10", "9", "5", "2", "10"])
seq_a = [env_a.reset(), env_a.step("stand")]

env_b = make_env("minimal_basic_strategy", ["9", "7", "7", "10"])
seq_b = [env_b.reset()]

seq_batch = encoder.encode_sequence_batch([seq_a, seq_b])

print("state_vector shape:", tuple(seq_batch["state_vector"].shape))
print("action_mask shape:", tuple(seq_batch["action_mask"].shape))
print("padding_mask shape:", tuple(seq_batch["padding_mask"].shape))
print("padding_mask:")
print(seq_batch["padding_mask"])
print("sequence_lengths:", seq_batch["metadata"]["sequence_lengths"])


state_vector shape: (2, 2, 40)
action_mask shape: (2, 2, 6)
padding_mask shape: (2, 2)
padding_mask:
tensor([[ True,  True],
        [ True, False]])
sequence_lengths: [2, 1]


In [17]:
# Chelper visual para ver qué parte del vector pertenece a cada módulo

def summarize_modules(encoded):
    meta = encoded["metadata"]
    for name, (start, end) in meta["module_slices"].items():
        tensor = encoded["state_vector"][start:end]
        print(
            f"{name:20s} | slice=({start:4d}, {end:4d}) | "
            f"shape={tuple(tensor.shape)} | nonzero={int(torch.count_nonzero(tensor))}"
        )

summarize_modules(encoded_default)


hand                 | slice=(   0,  185) | shape=(185,) | nonzero=8
other_hands          | slice=( 185,  885) | shape=(700,) | nonzero=0
hand_context         | slice=( 885,  890) | shape=(5,) | nonzero=2
insurance            | slice=( 890,  892) | shape=(2,) | nonzero=0
bet                  | slice=( 892,  893) | shape=(1,) | nonzero=1
rules                | slice=( 893,  910) | shape=(17,) | nonzero=9
observed_history     | slice=( 910,  924) | shape=(14,) | nonzero=3
discard_summary      | slice=( 924, 1068) | shape=(144,) | nonzero=9
temporal             | slice=(1068, 1089) | shape=(21,) | nonzero=9
